In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from torch_geometric.loader import DataLoader
import torch
from GNN.physics_aware_NN import ShardedQuantumCircuitGraphDataset
from GNN.training.utils import collect_dataset_indices


In [3]:
training_data_dir = "final/data/saturated"
index_path = collect_dataset_indices(
    training_data_dir,
    family="random",
)
print(f"Found {len(index_path)} samples in the dataset.")

Found 1 samples in the dataset.


In [4]:
print(p for p in index_path[:5])

<generator object <genexpr> at 0x0000022DE4DF8B80>


In [5]:
ds = ShardedQuantumCircuitGraphDataset(
    index_paths=index_path,
    split="target",
    target_variant="sre_density",
    cache_size=4,
)

Detected node feature dims: [17, 19, 21, 23]
Detected global feature dims: [13770, 22950]


In [6]:
from torch_geometric.data import Batch
import torch
from collections import defaultdict

def debug_random_batches(ds, batch_size=8, trials=200):
    for trial in range(trials):
        idxs = torch.randperm(len(ds))[:batch_size].tolist()
        samples = [ds[i] for i in idxs]

        try:
            _ = Batch.from_data_list(samples)
        except Exception as e:
            print("\nFAILED")
            print("trial:", trial)
            print("indices:", idxs)
            print("error:", repr(e))

            all_keys = sorted(set().union(*(set(d.keys()) for d in samples)))

            print("\nTensor shapes by key:")
            for key in all_keys:
                shapes = []
                for d in samples:
                    if hasattr(d, key):
                        value = getattr(d, key)
                        if torch.is_tensor(value):
                            shapes.append(tuple(value.shape))
                        else:
                            shapes.append(type(value).__name__)
                    else:
                        shapes.append("MISSING")

                unique = sorted(set(map(str, shapes)))
                if len(unique) > 1:
                    print(f"\nLIKELY PROBLEM: {key}")
                    for idx, shape in zip(idxs, shapes):
                        print(f"  idx={idx:6d} shape={shape}")

            return samples, idxs

    print("No failing random batch found.")
    return None, None

samples, idxs = debug_random_batches(ds, batch_size=8, trials=500)

No failing random batch found.


In [7]:
loader = DataLoader(ds, batch_size=8, shuffle=True)
batch = next(iter(loader))

print("num_graphs:", batch.num_graphs)
print("x:", batch.x.shape, batch.x.dtype)
print("edge_index:", batch.edge_index.shape, batch.edge_index.dtype)
print("global_features:", batch.global_features.shape, batch.global_features.dtype)
print("y:", batch.y.shape, batch.y[:8])
print("sre:", batch.sre.shape if hasattr(batch, "sre") else None)
print("raw_sre:", batch.raw_sre.shape if hasattr(batch, "raw_sre") else None)
print("n_qubits:", batch.n_qubits.shape if hasattr(batch, "n_qubits") else None)

assert batch.global_features.ndim == 2
assert batch.global_features.shape[0] == batch.num_graphs
assert torch.isfinite(batch.y).all()

num_graphs: 8
x: torch.Size([6453, 23]) torch.float32
edge_index: torch.Size([2, 7714]) torch.int64
global_features: torch.Size([8, 22950]) torch.float32
y: torch.Size([8]) tensor([0.4368, 0.8000, 0.6460, 0.5767, 0.6774, 0.8007, 0.7993, 0.6693])
sre: torch.Size([8])
raw_sre: torch.Size([8])
n_qubits: torch.Size([8])
